# AIG230 - Assignment 6: Natural Language Processing

In [ ]:
import random
from collections import Counter

import nltk
from nltk.corpus import brown

# Ensure Brown corpus is available
nltk.download("brown", quiet=True)

# ----- Special tokens -----
BOS = "<bos>"
EOS = "<eos>"
UNK = "<unk>"
SPECIAL_TOKENS = [UNK, BOS, EOS]

# ----- Config -----
SEED = 42
SPLIT_RATIOS = (0.8, 0.1, 0.1)  # train, valid, test
MIN_FREQ = 2
assert abs(sum(SPLIT_RATIOS) - 1.0) < 1e-8

# ----- D0: Load and preprocess (single shared pipeline) -----
def is_punctuation_only(token: str) -> bool:
    # Keep tokens that contain at least one letter or digit.
    # This removes punctuation-only tokens while preserving words like "don't".
    return not any(ch.isalnum() for ch in token)


def preprocess_sentence(sentence_tokens):
    cleaned = []
    for token in sentence_tokens:
        tok = token.lower()
        if is_punctuation_only(tok):
            continue
        cleaned.append(tok)
    return [BOS] + cleaned + [EOS]


raw_sentences = brown.sents(categories="news")
processed_sentences = [preprocess_sentence(sent) for sent in raw_sentences]

# Remove degenerate samples like [<bos>, <eos>] after punctuation filtering
processed_sentences = [s for s in processed_sentences if len(s) > 2]

# ----- D1: One shared split (sentence-level, deterministic) -----
rng = random.Random(SEED)
indices = list(range(len(processed_sentences)))
rng.shuffle(indices)

n_total = len(indices)
n_train = int(SPLIT_RATIOS[0] * n_total)
n_valid = int(SPLIT_RATIOS[1] * n_total)

train_idx = indices[:n_train] 
valid_idx = indices[n_train:n_train + n_valid]
test_idx = indices[n_train + n_valid:]

train_sentences = [processed_sentences[i] for i in train_idx]
valid_sentences = [processed_sentences[i] for i in valid_idx]
test_sentences = [processed_sentences[i] for i in test_idx]

# Token counts by split (before UNK mapping)
train_tokens_raw = [tok for sent in train_sentences for tok in sent]
valid_tokens_raw = [tok for sent in valid_sentences for tok in sent]
test_tokens_raw = [tok for sent in test_sentences for tok in sent]

# ----- D2: Vocabulary from training split only (shared) -----
train_counter = Counter(train_tokens_raw)

vocab = list(SPECIAL_TOKENS)
for token, freq in train_counter.items():
    if token in {UNK, BOS, EOS}:
        continue
    if freq >= MIN_FREQ:
        vocab.append(token)

token_to_id = {tok: i for i, tok in enumerate(vocab)}
id_to_token = {i: tok for tok, i in token_to_id.items()}


def map_sentence_to_vocab(sentence):
    return [tok if tok in token_to_id else UNK for tok in sentence]


train_sentences_mapped = [map_sentence_to_vocab(s) for s in train_sentences]
valid_sentences_mapped = [map_sentence_to_vocab(s) for s in valid_sentences]
test_sentences_mapped = [map_sentence_to_vocab(s) for s in test_sentences]

# Shared token streams after vocab mapping (for n-gram + RNN)
train_tokens = [tok for sent in train_sentences_mapped for tok in sent]
valid_tokens = [tok for sent in valid_sentences_mapped for tok in sent]
test_tokens = [tok for sent in test_sentences_mapped for tok in sent]

# ID streams for PyTorch RNN
def tokens_to_ids(tokens):
    return [token_to_id.get(tok, token_to_id[UNK]) for tok in tokens]


train_ids = tokens_to_ids(train_tokens)
valid_ids = tokens_to_ids(valid_tokens)
test_ids = tokens_to_ids(test_tokens)

# Sentence-level IDs if needed later
train_sentence_ids = [tokens_to_ids(s) for s in train_sentences_mapped]
valid_sentence_ids = [tokens_to_ids(s) for s in valid_sentences_mapped]
test_sentence_ids = [tokens_to_ids(s) for s in test_sentences_mapped]

# ----- Reports (D1 + D2) -----
print("D1 Report")
print(f"Sentences (train/valid/test): {len(train_sentences)} / {len(valid_sentences)} / {len(test_sentences)}")
print(f"Tokens    (train/valid/test): {len(train_tokens_raw)} / {len(valid_tokens_raw)} / {len(test_tokens_raw)}")
print(f"Vocab size (train-only): {len(vocab)}")
print()

print("D2 Report")
print(f"Chosen min_freq: {MIN_FREQ}")
print(f"Final vocab size: {len(vocab)}")
print("Top 20 most frequent tokens in train split:")
for tok, freq in train_counter.most_common(20):
    print(f"  {tok:>12} : {freq}")
print()

print(f"Special token IDs: UNK={token_to_id[UNK]}, BOS={token_to_id[BOS]}, EOS={token_to_id[EOS]}")
print("Example processed sentence:", train_sentences_mapped[0][:20])

D1 Report
Sentences (train/valid/test): 3688 / 461 / 462
Tokens    (train/valid/test): 77883 / 10104 / 9827
Vocab size (train-only): 5396

D2 Report
Chosen min_freq: 2
Final vocab size: 5396
Top 20 most frequent tokens in train split:
           the : 5073
         <bos> : 3688
         <eos> : 3688
            of : 2288
           and : 1734
             a : 1690
            to : 1682
            in : 1581
           for : 760
          that : 653
            is : 576
            on : 556
           was : 553
            he : 517
            at : 500
          with : 471
            as : 411
            be : 403
            by : 402
            it : 379

Special token IDs: UNK=0, BOS=1, EOS=2
Example processed sentence: ['<bos>', 'the', 'land', 'lies', 'ready', 'for', 'the', 'coming', '<unk>', '<eos>']
